# Dunnhumby Complete Journey - Exploratory Data Analysis
**Team ProAnalytics**

Explore the real Dunnhumby dataset before running the uplift modeling pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load all tables
transactions = pd.read_csv('data/transaction_data.csv')
demographics = pd.read_csv('data/hh_demographic.csv')
campaigns = pd.read_csv('data/campaign_table.csv')
campaign_desc = pd.read_csv('data/campaign_desc.csv')
coupons = pd.read_csv('data/coupon.csv')
coupon_redempt = pd.read_csv('data/coupon_redempt.csv')
causal_data = pd.read_csv('data/causal_data.csv')

print(f"Transactions: {len(transactions):,} rows")
print(f"Households: {transactions['household_key'].nunique():,}")
print(f"Products: {transactions['PRODUCT_ID'].nunique():,}")
print(f"Campaigns: {len(campaigns):,} exposures")
print(f"Coupon redemptions: {len(coupon_redempt):,}")

## 2. Transaction Analysis

In [ ]:
# Transaction overview
transactions.head()

In [ ]:
# Purchase frequency distribution
purchase_freq = transactions.groupby('household_key')['BASKET_ID'].nunique()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
purchase_freq.hist(bins=50, edgecolor='black')
plt.xlabel('Number of Purchases')
plt.ylabel('Number of Households')
plt.title('Purchase Frequency Distribution')

plt.subplot(1, 2, 2)
purchase_freq.plot(kind='box')
plt.ylabel('Number of Purchases')
plt.title('Purchase Frequency Box Plot')
plt.tight_layout()
plt.show()

print(f"Mean purchases per household: {purchase_freq.mean():.1f}")
print(f"Median purchases per household: {purchase_freq.median():.1f}")

In [ ]:
# Basket value distribution
basket_value = transactions.groupby('BASKET_ID')['SALES_VALUE'].sum()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
basket_value.hist(bins=50, edgecolor='black')
plt.xlabel('Basket Value ($)')
plt.ylabel('Frequency')
plt.title('Basket Value Distribution')

plt.subplot(1, 2, 2)
basket_value[basket_value < basket_value.quantile(0.95)].hist(bins=50, edgecolor='black')
plt.xlabel('Basket Value ($)')
plt.ylabel('Frequency')
plt.title('Basket Value Distribution (95th percentile)')
plt.tight_layout()
plt.show()

print(f"Mean basket value: ${basket_value.mean():.2f}")
print(f"Median basket value: ${basket_value.median():.2f}")

## 3. Demographics Analysis

In [ ]:
demographics.head()

In [ ]:
# Age distribution
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
demographics['AGE_DESC'].value_counts().plot(kind='bar', edgecolor='black')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.title('Age Distribution')
plt.xticks(rotation=45)

plt.subplot(1, 3, 2)
demographics['INCOME_DESC'].value_counts().plot(kind='bar', edgecolor='black')
plt.xlabel('Income Group')
plt.ylabel('Count')
plt.title('Income Distribution')
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
demographics['HH_COMP_DESC'].value_counts().plot(kind='bar', edgecolor='black')
plt.xlabel('Household Composition')
plt.ylabel('Count')
plt.title('Household Composition')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 4. Campaign Analysis

In [ ]:
# Campaign exposure
campaign_full = campaigns.merge(campaign_desc, on='CAMPAIGN')

print(f"Total campaign exposures: {len(campaign_full):,}")
print(f"Unique households exposed: {campaign_full['household_key'].nunique():,}")
print(f"\nCampaign types:")
print(campaign_full['CAMPAIGN_TYPE'].value_counts())

In [ ]:
# Campaign exposure by type
plt.figure(figsize=(10, 5))
campaign_full['CAMPAIGN_TYPE'].value_counts().plot(kind='bar', edgecolor='black')
plt.xlabel('Campaign Type')
plt.ylabel('Number of Exposures')
plt.title('Campaign Exposures by Type')
plt.xticks(rotation=0)
plt.show()

## 5. Coupon Redemption Analysis

In [ ]:
# Coupon redemption rate
redemptions_per_hh = coupon_redempt.groupby('household_key').size()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
redemptions_per_hh.hist(bins=30, edgecolor='black')
plt.xlabel('Number of Coupons Redeemed')
plt.ylabel('Number of Households')
plt.title('Coupon Redemption Distribution')

plt.subplot(1, 2, 2)
redemptions_per_hh.plot(kind='box')
plt.ylabel('Number of Coupons Redeemed')
plt.title('Coupon Redemption Box Plot')
plt.tight_layout()
plt.show()

print(f"Households that redeemed coupons: {len(redemptions_per_hh):,}")
print(f"Mean redemptions per household: {redemptions_per_hh.mean():.1f}")
print(f"Median redemptions per household: {redemptions_per_hh.median():.1f}")

## 6. Treatment vs Control Comparison

In [ ]:
# Define treatment (TypeA campaigns) vs control
treated_hh = campaign_full[campaign_full['CAMPAIGN_TYPE'] == 'TypeA']['household_key'].unique()
all_hh = transactions['household_key'].unique()
control_hh = np.setdiff1d(all_hh, treated_hh)

print(f"Treatment group: {len(treated_hh):,} households ({len(treated_hh)/len(all_hh)*100:.1f}%)")
print(f"Control group: {len(control_hh):,} households ({len(control_hh)/len(all_hh)*100:.1f}%)")

In [ ]:
# Compare spending between treatment and control
hh_spend = transactions.groupby('household_key')['SALES_VALUE'].sum()
treated_spend = hh_spend[hh_spend.index.isin(treated_hh)]
control_spend = hh_spend[hh_spend.index.isin(control_hh)]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist([treated_spend, control_spend], bins=50, label=['Treated', 'Control'], alpha=0.7, edgecolor='black')
plt.xlabel('Total Spend ($)')
plt.ylabel('Frequency')
plt.title('Spending Distribution: Treatment vs Control')
plt.legend()

plt.subplot(1, 2, 2)
data_to_plot = [treated_spend, control_spend]
plt.boxplot(data_to_plot, labels=['Treated', 'Control'])
plt.ylabel('Total Spend ($)')
plt.title('Spending Comparison')
plt.tight_layout()
plt.show()

print(f"Mean spend (treated): ${treated_spend.mean():.2f}")
print(f"Mean spend (control): ${control_spend.mean():.2f}")
print(f"Difference: ${treated_spend.mean() - control_spend.mean():.2f}")

## 7. Summary Statistics

In [ ]:
# Overall summary
summary = {
    'Total Households': len(all_hh),
    'Total Transactions': len(transactions),
    'Total Revenue': f"${transactions['SALES_VALUE'].sum():,.2f}",
    'Avg Basket Value': f"${basket_value.mean():.2f}",
    'Avg Purchases per HH': f"{purchase_freq.mean():.1f}",
    'Campaign Exposures': len(campaign_full),
    'Coupon Redemptions': len(coupon_redempt),
    'Treatment Rate': f"{len(treated_hh)/len(all_hh)*100:.1f}%"
}

pd.DataFrame(summary.items(), columns=['Metric', 'Value'])